In [ ]:
import chromadb
import ollama

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

pdf_path = "C:/Users/saran/simple_fruits_guide.pdf"
chat_model = "llama3.2"

def load_pdf(pdf_path):
  reader = PdfReader(pdf_path)
  text = ""
  for page in reader.pages:
    text += (page.extract_text() or "") + "\n"
  return text

def create_chunks(text, size=800, overlap=100):
  chunks = []
  start = 0
  while start < len(text):
    chunks.append(text[start:start + size])
    start += size - overlap
  return chunks

def create_database(chunks, model):
  client = chromadb.EphemeralClient()
  db = client.create_collection("pdf")
  db.add(
       ids = [str(i) for i in range(len(chunks))],
       documents = chunks,
       embeddings = model.encode(chunks).tolist()
     )
  return db

def find_context(db, model, question, top_k=3):
  q_embedding = model.encode([question]).tolist()
  result = db.query(query_embeddings=q_embedding, n_results=top_k)
  return "\n\n".join(result["documents"][0])

def ask_model(question, context):
            prompt = f"""Answer using ONLY the context below. If the answer is not there, say "I could not find that in the document."
          Context: {context}
          Question: {question}
          """
            for part in ollama.chat(
                model = chat_model,
                messages = [{"role": "user", "content": prompt}],
                stream = True
             ):
                print(part["message"]["content"], end="", flush=True)
            print()

def main():
  model = SentenceTransformer("all-MiniLM-L6-v2")
  text = load_pdf(pdf_path)
  chunks = create_chunks(text)
  db = create_database(chunks, model)
  while True:
    question = input("\nEnter your question: ")
    if question.lower() in ("quit", "exit"):
      break
    if not question:
      continue
    context = find_context(db, model, question)
    print("\nResponse: ", end="")
    ask_model(question, context)

if __name__ == "__main__":
  main()